# BRFSS Prevalence Data: Dataset Build (v1.1)

Filters the full BRFSS file down to what the web app needs and writes two files into `data/`:

- `BRFSS_AppData_v1.1.csv`: the filtered rows, with every response kept
- `BRFSS_Lookup_v1.1.csv`: one row per health measure, which the app reads to fill its dropdown

This notebook expects to sit in the folder that contains the `data` folder.

**Decisions applied**

| Choice | Kept |
|---|---|
| Health measures | Asked in all 15 years, not in the Demographics class (23 measures) |
| Locations | 50 states and DC, plus UW (median of states and DC) for a reference line |
| Break-outs | Overall, Sex, the usual 6 age groups, Education, and White, Black, Hispanic |
| Dropped | Household Income, other race groups, unusual age groups, GU, PR, VI, US |

**Changes from v1.0**

- BMI and health insurance response labels merged (Section 6), with a note for each in the lookup table
- Default responses filled for 4 of the 6 multi-category measures (Section 7)
- Pregnancy-related diabetes missing values split by Female and Male (Section 8)
- Missing rates recomputed on the default-response rows only, which are the rows the app plots (Section 9)
- The v1.0 near-duplicate check now runs after the merge and must find none

**Sections**

1. Setup and load
2. Keep the 23 health measures
3. Short labels
4. Keep locations
5. Keep break-outs
6. Merge response labels
7. Default responses
8. Diabetes missing values
9. Missing values for the rows the app plots
10. Write the two files

## 1. Setup and load

In [1]:
import re
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

# show every column instead of collapsing the middle ones into "..."
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

In [2]:
data_dir = Path("data")
data_path = data_dir / "BRFSS_Prevalence_Data.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"File not found at {data_path.resolve()}. "
        "Check that the notebook sits in the folder that contains the data folder."
    )

keep_cols = [
    "Year",
    "Locationabbr",
    "Locationdesc",
    "Class",
    "Topic",
    "Question",
    "Response",
    "Break_Out_Category",
    "Break_Out",
    "Sample_Size",
    "Data_value",
    "Confidence_limit_Low",
    "Confidence_limit_High",
]

start = time.perf_counter()
# thousands="," reads numbers such as 1,025 as numbers, see BRFSS_DataExploration_v1.1 Section 2 for all read options
df_full = pd.read_csv(data_path, usecols=keep_cols, thousands=",", low_memory=False)
df_full = df_full[keep_cols]
print(f"Loaded {len(df_full):,} rows in {time.perf_counter() - start:,.1f} seconds")

Loaded 3,211,643 rows in 12.4 seconds


In [3]:
def show_all(table):
    """
    Display a table with no limit on the number of rows shown

    Parameters
    ----------
    table : pandas.DataFrame or pandas.Series
        Table to display
    """
    # option_context lifts the row limit only inside the with block
    with pd.option_context("display.max_rows", None):
        display(table)

## 2. Keep the 23 health measures

The rule is computed from the data rather than typed as a list:

1. The question has rows in every year of the file (15 years)
2. The question is not in the Demographics class (age, sex, race, education, income, marital status,
   employment, number of children)

The `assert` stops the notebook if the result is not 23, so a future export that changes the
question set does not slip through unnoticed.

In [4]:
n_years_total = df_full["Year"].nunique()

# for each question, the number of distinct years with rows
years_per_q = df_full.groupby("Question")["Year"].nunique()
all_year_qs = years_per_q.index[years_per_q == n_years_total]

# the Class of each question, drop_duplicates keeps one row per question
q_class = df_full[["Question", "Class"]].drop_duplicates().set_index("Question")["Class"]

kept_questions = [q for q in all_year_qs if q_class[q] != "Demographics"]

print(f"Questions asked in all {n_years_total} years: {len(all_year_qs)}")
print(f"Of those, outside Demographics: {len(kept_questions)}")
assert len(kept_questions) == 23, "Expected 23 health measures, check the rule above"

# isin keeps the rows whose Question is in the list
df_q = df_full[df_full["Question"].isin(kept_questions)].copy()
print(f"Rows after keeping the 23 measures: {len(df_q):,}")

Questions asked in all 15 years: 31
Of those, outside Demographics: 23
Rows after keeping the 23 measures: 1,112,906


## 3. Short labels

Each short label is matched to its question by the start of the question text. The CDC text is long
and some of it ends in "(variable calculated from one or more BRFSS questions)", so matching on the
start avoids typing the full text.

The checks below confirm that every label matches exactly one of the 23 questions and that every
question receives a label.

In [5]:
label_prefixes = {
    "Health insurance (ages 18-64)": "Adults aged 18-64 who have any kind of health care coverage",
    "Arthritis": "Adults who have been told they have arthritis",
    "Current smoking": "Adults who are current smokers",
    "Flu shot (ages 65+)": "Adults aged 65+ who have had a flu shot",
    "Pneumonia vaccine (ages 65+)": "Adults aged 65+ who have ever had a pneumonia vaccination",
    "Asthma (ever)": "Adults who have ever been told they have asthma",
    "Asthma (current)": "Adults who have been told they currently have asthma",
    "Any alcohol (past 30 days)": "Adults who have had at least one drink of alcohol",
    "Binge drinking": "Binge drinkers",
    "Heart attack": "Ever told you had a heart attack",
    "Stroke": "Ever told you had a stroke",
    "Coronary heart disease": "Ever told you had angina or coronary heart disease",
    "Depression": "Ever told you that you have a form of depression",
    "Diabetes": "Have you ever been told by a doctor that you have diabetes",
    "Smoking status": "Four Level Smoking Status",
    "Weight category (BMI)": "Weight classification by Body Mass Index",
    "General health rating": "How is your general health",
    "Fair or poor health": "Health Status (variable calculated",
    "Skin cancer": "Ever told you had skin cancer",
    "COPD": "Ever told you have COPD",
    "Kidney disease": "Ever told you have kidney disease",
    "Any exercise (past month)": "During the past month, did you participate in any physical activities",
    "Other cancer": "Ever told you had any other types of cancer",
}

question_to_label = {}
for label, prefix in label_prefixes.items():
    # startswith checks whether the question text begins with the prefix
    matches = [q for q in kept_questions if q.startswith(prefix)]
    assert len(matches) == 1, f"{label!r} matched {len(matches)} questions, expected 1"
    question_to_label[matches[0]] = label

unlabeled = [q for q in kept_questions if q not in question_to_label]
assert not unlabeled, f"Questions with no label: {unlabeled}"

# map looks up each row's question in the dictionary and returns the matching label
df_q["short_label"] = df_q["Question"].map(question_to_label)

label_table = pd.DataFrame(
    {"short_label": list(question_to_label.values()), "question": list(question_to_label.keys())}
).sort_values("short_label")
show_all(label_table)

,short_label,question
7,Any alcohol (past 30 days),Adults who have had at least one drink of alcohol within the past 30 days
21,Any exercise (past month),"During the past month, did you participate in any physical activities? (variable calculated from..."
1,Arthritis,Adults who have been told they have arthritis (variable calculated from one or more BRFSS questi...
6,Asthma (current),Adults who have been told they currently have asthma (variable calculated from one or more BRFSS...
5,Asthma (ever),Adults who have ever been told they have asthma (variable calculated from one or more BRFSS ques...
8,Binge drinking,"Binge drinkers (males having five or more drinks on one occasion, females having four or more dr..."
19,COPD,Ever told you have COPD?
11,Coronary heart disease,Ever told you had angina or coronary heart disease?
2,Current smoking,Adults who are current smokers (variable calculated from one or more BRFSS questions)
12,Depression,Ever told you that you have a form of depression?


## 4. Keep locations

The 50 states and DC, plus UW. UW is "All States and DC (median)", which the app can offer as a
reference line. UW rows exist only for the Overall break-out and have no confidence limits.

Dropped: GU, PR, VI (territories) and US (median that includes territories).

In [6]:
drop_locations = ["GU", "PR", "VI", "US"]

# ~ means "not", so the filter keeps rows whose location is not in the drop list
df_loc = df_q[~df_q["Locationabbr"].isin(drop_locations)].copy()

kept_locs = sorted(df_loc["Locationabbr"].unique())
state_locs = [loc for loc in kept_locs if loc != "UW"]
print(f"States and DC kept: {len(state_locs)}")
print(f"UW kept: {'UW' in kept_locs}")
assert len(state_locs) == 51, "Expected 50 states and DC"
print(f"Rows after location filter: {len(df_loc):,}")

States and DC kept: 51
UW kept: True
Rows after location filter: 1,062,643


## 5. Keep break-outs

Each category maps to the list of groups kept within it. Household Income is not in the dictionary,
so all its rows are dropped.

In [7]:
keep_breakouts = {
    "Overall": ["Overall"],
    "Sex": ["Female", "Male"],
    "Age Group": ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "Education Attained": ["College graduate", "H.S. or G.E.D.", "Less than H.S.", "Some post-H.S."],
    "Race/Ethnicity": ["White, non-Hispanic", "Black, non-Hispanic", "Hispanic"],
}

# build one True or False per row: True when the row's category and group are both in keep_breakouts
breakout_ok = pd.Series(False, index=df_loc.index)
for category, groups in keep_breakouts.items():
    in_this = (df_loc["Break_Out_Category"] == category) & df_loc["Break_Out"].isin(groups)
    # | means "or", so a row stays True once any category matches
    breakout_ok = breakout_ok | in_this

df_app = df_loc[breakout_ok].copy()
print(f"Rows after break-out filter: {len(df_app):,}")

# confirm every listed group was found in the data, a typo in a label would show here as missing
found = df_app.groupby("Break_Out_Category")["Break_Out"].unique()
for category, groups in keep_breakouts.items():
    missing_groups = set(groups) - set(found.get(category, []))
    print(f"{category}: {'all groups found' if not missing_groups else f'NOT FOUND {missing_groups}'}")

Rows after break-out filter: 646,725
Overall: all groups found
Sex: all groups found
Age Group: all groups found
Education Attained: all groups found
Race/Ethnicity: all groups found


## 6. Merge response labels

v1.0 Section 6 found two measures whose labels changed over time while meaning the same thing:

| Measure | Old label | Years | Merged into |
|---|---|---|---|
| Weight category (BMI) | 4 labels with lowercase "bmi" | 2015 to 2017 | The same labels with "BMI" |
| Health insurance (ages 18-64) | "Have some form of health insurance" | 2021 to 2025 | Yes |
| Health insurance (ages 18-64) | "Do not have some form of health insurance" | 2021 to 2025 | No |

Two labels can be merged only if they never appear in the same year. The first cell checks that
in code before changing anything.

In [8]:
label_merges = {
    "Weight category (BMI)": {
        "Normal Weight (bmi 18.5-24.9)": "Normal Weight (BMI 18.5-24.9)",
        "Obese (bmi 30.0 - 99.8)": "Obese (BMI 30.0 - 99.8)",
        "Overweight (bmi 25.0-29.9)": "Overweight (BMI 25.0-29.9)",
        "Underweight (bmi 12.0-18.4)": "Underweight (BMI 12.0-18.4)",
    },
    "Health insurance (ages 18-64)": {
        "Have some form of health insurance": "Yes",
        "Do not have some form of health insurance": "No",
    },
}

for measure, mapping in label_merges.items():
    rows = df_app[df_app["short_label"] == measure]
    for old_label, new_label in mapping.items():
        # set() collects the distinct years for each label, & keeps the years the two sets share
        old_years = set(rows.loc[rows["Response"] == old_label, "Year"])
        new_years = set(rows.loc[rows["Response"] == new_label, "Year"])
        shared = old_years & new_years
        assert not shared, f"{measure}: {old_label!r} and {new_label!r} both appear in {sorted(shared)}"
        print(f"{measure}: {old_label!r} ({len(old_years)} years) and {new_label!r} ({len(new_years)} years) never overlap")

Weight category (BMI): 'Normal Weight (bmi 18.5-24.9)' (3 years) and 'Normal Weight (BMI 18.5-24.9)' (12 years) never overlap
Weight category (BMI): 'Obese (bmi 30.0 - 99.8)' (3 years) and 'Obese (BMI 30.0 - 99.8)' (12 years) never overlap
Weight category (BMI): 'Overweight (bmi 25.0-29.9)' (3 years) and 'Overweight (BMI 25.0-29.9)' (12 years) never overlap
Weight category (BMI): 'Underweight (bmi 12.0-18.4)' (3 years) and 'Underweight (BMI 12.0-18.4)' (12 years) never overlap
Health insurance (ages 18-64): 'Have some form of health insurance' (5 years) and 'Yes' (10 years) never overlap
Health insurance (ages 18-64): 'Do not have some form of health insurance' (5 years) and 'No' (10 years) never overlap


In [9]:
for measure, mapping in label_merges.items():
    is_measure = df_app["short_label"] == measure
    before = df_app.loc[is_measure, "Response"].copy()
    # replace swaps each old label for the matching new label, labels not in the mapping stay unchanged
    df_app.loc[is_measure, "Response"] = df_app.loc[is_measure, "Response"].replace(mapping)
    # count rows whose label changed, fillna stops missing responses from counting as changes
    n_changed = (before.fillna("") != df_app.loc[is_measure, "Response"].fillna("")).sum()
    print(f"{measure}: {n_changed:,} rows relabeled")

Weight category (BMI): 9,804 rows relabeled
Health insurance (ages 18-64): 7,450 rows relabeled


**Check after the merge.** The v1.0 near-duplicate check runs again and must find nothing. Then each
measure with more than two responses gets a response-by-year table. For the two merged measures, every
response must now appear in all 15 years.

In [10]:
def normalize_label(text):
    """
    Lowercase a label and collapse repeated whitespace to one space

    Parameters
    ----------
    text : str
        Original response label

    Returns
    -------
    str
        Label in lowercase with single spaces and no spaces at the ends
    """
    # re.sub replaces every run of whitespace (\s+) with one space
    return re.sub(r"\s+", " ", text).strip().lower()


resp_pairs = df_app[["short_label", "Response"]].dropna().drop_duplicates().copy()
resp_pairs["normalized"] = resp_pairs["Response"].map(normalize_label)
variant_counts = resp_pairs.groupby(["short_label", "normalized"])["Response"].transform("size")
n_near = int((variant_counts > 1).sum())
print(f"Labels that differ only in capitalization or spacing: {n_near}")
assert n_near == 0, "Near-duplicate labels remain after the merge"

Labels that differ only in capitalization or spacing: 0


In [11]:
n_resp = df_app.groupby("short_label")["Response"].nunique()
multi_resp = n_resp[n_resp > 2].index.tolist()
print(f"Measures with more than two responses: {multi_resp}")

for label in multi_resp:
    rows = df_app[df_app["short_label"] == label]
    print(f"\n{label}")
    show_all(pd.crosstab(rows["Response"], rows["Year"]))

for measure in label_merges:
    rows = df_app[df_app["short_label"] == measure]
    years_per_resp = rows.groupby("Response")["Year"].nunique()
    assert (years_per_resp == n_years_total).all(), f"{measure}: some responses missing years after merge"
    print(f"{measure}: every response present in all {n_years_total} years")

Measures with more than two responses: ['Diabetes', 'General health rating', 'Smoking status', 'Weight category (BMI)']

Diabetes


Year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Response,,,,,,,,,,,,,,,
No,817,817,817,817,817,817,817,817,801,817,801,817,785,801,769
"No, pre-diabetes or borderline diabetes",817,785,817,789,817,817,817,817,801,817,801,817,785,801,769
Yes,817,815,817,816,817,817,817,817,801,817,801,817,785,801,769
"Yes, pregnancy-related",766,735,766,738,817,817,817,817,801,817,801,817,785,801,769



General health rating


Year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Response,,,,,,,,,,,,,,,
Excellent,817,816,817,817,817,817,817,817,801,817,801,817,785,801,769
Fair,817,817,817,816,817,817,817,817,801,817,801,817,785,801,769
Good,817,817,817,817,817,817,817,817,801,817,801,817,785,801,769
Poor,817,812,817,811,817,817,817,817,801,817,801,817,785,801,769
Very good,817,817,817,817,817,817,817,817,801,817,801,817,785,801,769



Smoking status


Year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Response,,,,,,,,,,,,,,,
Former smoker,817,817,817,817,817,817,817,817,801,817,801,817,785,801,769
Never smoked,817,817,817,817,817,817,817,817,801,817,801,817,785,801,769
Smoke everyday,817,817,817,816,817,817,817,817,801,817,801,817,785,801,769
Smoke some days,817,814,817,814,817,817,817,817,801,817,801,817,785,801,769



Weight category (BMI)


Year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Response,,,,,,,,,,,,,,,
Normal Weight (BMI 18.5-24.9),817,816,817,817,817,817,817,817,801,817,801,817,785,801,769
Obese (BMI 30.0 - 99.8),817,817,817,817,817,817,817,817,801,817,801,817,785,801,769
Overweight (BMI 25.0-29.9),817,817,817,817,817,817,817,817,801,817,801,817,785,801,769
Underweight (BMI 12.0-18.4),817,805,817,802,817,817,817,817,801,817,801,817,785,801,769


Weight category (BMI): every response present in all 15 years
Health insurance (ages 18-64): every response present in all 15 years


**Notes for the app.** Each merge gets a note in the lookup table, so the app can show it under the plot
and the README can repeat it.

In [12]:
measure_notes = {
    "Weight category (BMI)": (
        "Label capitalization differed in 2015 to 2017. Labels merged, values unchanged."
    ),
    "Health insurance (ages 18-64)": (
        "Response wording changed in 2021. Responses merged into Yes and No. "
        "A jump between 2020 and 2021 may reflect a CDC change in how the measure is calculated."
    ),
}

## 7. Default responses

Most plots show one number per measure, so each measure names the response to plot. For the 17 Yes/No
measures it is "Yes". For the 6 measures with more categories it is chosen below. General health rating
and Smoking status stay blank until a choice is made.

The check confirms every chosen default is a real response of that measure, so a typo cannot slip through.

In [13]:
default_choices = {
    "Diabetes": "Yes",
    "Fair or poor health": "Fair or Poor Health",
    "Health insurance (ages 18-64)": "Yes",
    "Weight category (BMI)": "Obese (BMI 30.0 - 99.8)",
    "General health rating": "",
    "Smoking status": "",
}

responses_by_measure = df_app.groupby("short_label")["Response"].apply(lambda r: set(r.dropna()))

default_response = {}
for label, responses in responses_by_measure.items():
    if label in default_choices:
        default_response[label] = default_choices[label]
    elif responses == {"Yes", "No"}:
        default_response[label] = "Yes"
    else:
        default_response[label] = ""

for label, choice in default_response.items():
    assert choice == "" or choice in responses_by_measure[label], f"{label}: {choice!r} is not a response"

blank_defaults = [label for label, choice in default_response.items() if choice == ""]
print(f"Measures with a default: {len(default_response) - len(blank_defaults)}")
print(f"Measures still blank: {blank_defaults}")

Measures with a default: 21
Measures still blank: ['General health rating', 'Smoking status']


## 8. Diabetes missing values

A missing `Data_value` means CDC did not report an estimate for that row, not that the answer was No.
v1.0 found pregnancy-related diabetes missing in 62.3% of rows, and 44.3% within the Sex category.
The last table splits that 44.3% into Female and Male.

In [14]:
diab = df_app[df_app["short_label"] == "Diabetes"].copy()
diab["value_missing"] = diab["Data_value"].isna()

print("Percent missing by response:")
print((diab.groupby("Response")["value_missing"].mean() * 100).round(1).to_string())

print("\nPercent missing by response and break-out category:")
miss_diab = pd.crosstab(diab["Response"], diab["Break_Out_Category"], values=diab["value_missing"], aggfunc="mean")
show_all((miss_diab * 100).round(1))

Percent missing by response:
Response
No                                          1.0
No, pre-diabetes or borderline diabetes    34.2
Yes                                        12.6
Yes, pregnancy-related                     62.3

Percent missing by response and break-out category:


Break_Out_Category,Age Group,Education Attained,Overall,Race/Ethnicity,Sex
Response,,,,,
No,0.0,0.0,0.0,5.1,0.0
"No, pre-diabetes or borderline diabetes",52.3,23.1,0.1,45.7,3.0
Yes,24.7,0.3,0.0,17.7,0.0
"Yes, pregnancy-related",77.4,62.1,3.9,62.9,44.3


In [15]:
preg_sex = diab[(diab["Response"] == "Yes, pregnancy-related") & (diab["Break_Out_Category"] == "Sex")]

preg_table = preg_sex.groupby("Break_Out").agg(
    rows=("value_missing", "size"),
    rows_missing=("value_missing", "sum"),
)
preg_table["pct_missing"] = (preg_table["rows_missing"] / preg_table["rows"] * 100).round(1)
print("Pregnancy-related diabetes, Sex category:")
print(preg_table.to_string())

Pregnancy-related diabetes, Sex category:
           rows  rows_missing  pct_missing
Break_Out                                 
Female      757            27          3.6
Male        553           553        100.0


## 9. Missing values for the rows the app plots

v1.0 counted every response. The app plots only the default response, so these rates count only those
rows. Measures with a blank default are left out until a default is chosen. UW is left out of the
state table since it is a median, not a state.

In [16]:
# map gives each row the default response of the row's measure, then the filter keeps rows matching that default
df_app["default_for_row"] = df_app["short_label"].map(default_response)
df_plot = df_app[(df_app["default_for_row"] != "") & (df_app["Response"] == df_app["default_for_row"])].copy()
df_plot["value_missing"] = df_plot["Data_value"].isna()

print(f"Rows the app plots: {len(df_plot):,}")
print(f"Measures left out (blank default): {blank_defaults}")
print(f"Overall percent missing: {df_plot['value_missing'].mean() * 100:.1f}")

print("\nBy health measure:")
print((df_plot.groupby("short_label")["value_missing"].mean() * 100).round(1).sort_values(ascending=False).to_string())

Rows the app plots: 244,631
Measures left out (blank default): ['General health rating', 'Smoking status']
Overall percent missing: 10.8

By health measure:
short_label
Skin cancer                      29.1
Coronary heart disease           27.3
Stroke                           26.2
Kidney disease                   25.6
Heart attack                     24.7
Other cancer                     17.9
COPD                             15.5
Diabetes                         12.6
Pneumonia vaccine (ages 65+)     10.3
Flu shot (ages 65+)              10.0
Arthritis                         5.3
Asthma (current)                  4.0
Current smoking                   2.9
Binge drinking                    2.8
Fair or poor health               2.6
Asthma (ever)                     2.5
Depression                        1.9
Health insurance (ages 18-64)     1.4
Weight category (BMI)             1.4
Any alcohol (past 30 days)        1.1
Any exercise (past month)         1.0


In [17]:
miss_group = (
    df_plot.groupby(["Break_Out_Category", "Break_Out"])["value_missing"]
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)
print("By break-out group:")
print(miss_group.to_string())

By break-out group:
Break_Out_Category  Break_Out          
Age Group           18-24                  44.6
Race/Ethnicity      Hispanic               35.8
Age Group           25-34                  33.5
Race/Ethnicity      Black, non-Hispanic    32.9
Age Group           35-44                  19.6
Education Attained  Less than H.S.          5.5
Age Group           45-54                   3.5
                    55-64                   0.3
Education Attained  H.S. or G.E.D.          0.2
                    Some post-H.S.          0.2
                    College graduate        0.1
Age Group           65+                     0.0
Overall             Overall                 0.0
Race/Ethnicity      White, non-Hispanic     0.0
Sex                 Female                  0.0
                    Male                    0.0


In [18]:
plot_states = df_plot[df_plot["Locationabbr"] != "UW"]
miss_state = (plot_states.groupby("Locationabbr")["value_missing"].mean() * 100).round(1).sort_values(ascending=False)
print("By state (highest first):")
print(miss_state.to_string())

By state (highest first):
Locationabbr
SD    21.3
AK    19.4
NH    18.6
VT    17.9
WY    17.7
ND    17.5
ID    16.4
MT    16.3
NV    15.4
DC    15.4
ME    14.9
HI    13.7
OR    13.6
WV    13.1
WI    12.2
NM    12.0
DE    11.6
MS    11.6
RI    11.5
IA    11.3
AR    10.9
TN    10.7
AL     9.8
KY     9.8
PA     9.7
IL     9.6
LA     9.5
MO     9.3
UT     9.2
AZ     9.2
SC     8.8
NC     8.8
VA     8.6
GA     8.5
OK     8.5
MI     7.7
OH     7.7
CT     7.6
MA     7.5
IN     7.4
NE     7.4
MN     7.3
NJ     7.3
MD     7.0
WA     6.6
CO     6.6
TX     6.4
FL     6.2
CA     5.6
KS     5.4
NY     4.8


## 10. Write the two files

**App data.** The filtered rows with the merged labels: all 13 original columns plus `short_label`.
Every response is kept, so a Response dropdown or category breakdown can use the same file.

**Lookup table.** One row per health measure:

| Column | Holds |
|---|---|
| short_label | Name shown in the app dropdown |
| question | Full CDC question text |
| topic, class | CDC categories |
| default_response | Response the app plots for single-number views, blank if not chosen yet |
| all_responses | Every response for the measure after the merge, separated by " \| " |
| notes | Note to show under the plot, blank for most measures |

In [19]:
app_cols = keep_cols + ["short_label"]
app_path = data_dir / "BRFSS_AppData_v1.1.csv"
# index=False leaves out the row numbers pandas would otherwise write as an extra first column
df_app[app_cols].to_csv(app_path, index=False)

app_mb = app_path.stat().st_size / 1024**2
print(f"Wrote {app_path}: {len(df_app):,} rows, {app_mb:,.1f} MB")

Wrote data\BRFSS_AppData_v1.1.csv: 646,725 rows, 119.8 MB


In [20]:
def response_summary(responses):
    """
    Join the distinct non-missing responses of one measure into a single string

    Parameters
    ----------
    responses : pandas.Series
        Response values for the rows of one measure

    Returns
    -------
    str
        Distinct responses in alphabetical order, separated by " | "
    """
    return " | ".join(sorted(responses.dropna().unique()))


lookup = (
    df_app.groupby(["short_label", "Question", "Topic", "Class"], as_index=False)["Response"]
    .agg(response_summary)
    .rename(columns={"Question": "question", "Topic": "topic", "Class": "class", "Response": "all_responses"})
)
lookup["default_response"] = lookup["short_label"].map(default_response)
# measures with no note get an empty string instead of NaN
lookup["notes"] = lookup["short_label"].map(measure_notes).fillna("")

lookup = lookup[["short_label", "question", "topic", "class", "default_response", "all_responses", "notes"]]
lookup = lookup.sort_values("short_label")

lookup_path = data_dir / "BRFSS_Lookup_v1.1.csv"
lookup.to_csv(lookup_path, index=False)
print(f"Wrote {lookup_path}: {len(lookup)} rows")
show_all(lookup)

Wrote data\BRFSS_Lookup_v1.1.csv: 23 rows


,short_label,question,topic,class,default_response,all_responses,notes
0,Any alcohol (past 30 days),Adults who have had at least one drink of alcohol within the past 30 days,Alcohol Consumption,Alcohol Consumption,Yes,No | Yes,
1,Any exercise (past month),"During the past month, did you participate in any physical activities? (variable calculated from...",Exercise,Physical Activity,Yes,No | Yes,
2,Arthritis,Adults who have been told they have arthritis (variable calculated from one or more BRFSS questi...,Arthritis,Chronic Health Indicators,Yes,No | Yes,
3,Asthma (current),Adults who have been told they currently have asthma (variable calculated from one or more BRFSS...,Asthma,Chronic Health Indicators,Yes,No | Yes,
4,Asthma (ever),Adults who have ever been told they have asthma (variable calculated from one or more BRFSS ques...,Asthma,Chronic Health Indicators,Yes,No | Yes,
5,Binge drinking,"Binge drinkers (males having five or more drinks on one occasion, females having four or more dr...",Binge Drinking,Alcohol Consumption,Yes,No | Yes,
6,COPD,Ever told you have COPD?,COPD,Chronic Health Indicators,Yes,No | Yes,
7,Coronary heart disease,Ever told you had angina or coronary heart disease?,Cardiovascular Disease,Chronic Health Indicators,Yes,No | Yes,
8,Current smoking,Adults who are current smokers (variable calculated from one or more BRFSS questions),Current Smoker Status,Tobacco Use,Yes,No | Yes,
9,Depression,Ever told you that you have a form of depression?,Depression,Chronic Health Indicators,Yes,No | Yes,
